In [ ]:
# ======================================
# 📚 导入依赖
# ======================================
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevance,
    context_precision,
    context_recall,
)
from ragas.llms import HuggingFacePipeline
from transformers import pipeline

# ======================================
# 🧩 Step 1. 准备示例数据
# ======================================
data = {
    "question": [
        "Who wrote the novel 1984?",
        "What is the capital of France?",
        "When did the Apollo 11 mission land on the Moon?"
    ],
    "contexts": [
        [
            "George Orwell was an English writer and journalist.",
            "He is best known for his novels 1984 and Animal Farm."
        ],
        [
            "Paris is the capital and most populous city of France.",
            "It is known for landmarks such as the Eiffel Tower."
        ],
        [
            "Apollo 11 was the first manned mission to land on the Moon.",
            "The landing occurred on July 20, 1969, with Neil Armstrong and Buzz Aldrin."
        ]
    ],
    "answers": [
        ["George Orwell"],
        ["Paris"],
        ["July 20, 1969"]
    ],
    "generated_answer": [
        "1984 was written by George Orwell.",
        "The capital of France is Paris.",
        "Apollo 11 landed on the Moon in 1969."
    ]
}

dataset = Dataset.from_dict(data)

print("✅ 示例数据预览：")
print(dataset)

# ======================================
# 🧠 Step 2. 定义 LLM（可选）
# 默认会用 OpenAI 或 HF Hub 模型
# 这里演示使用 HuggingFace 本地 pipeline
# ======================================
hf_pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=hf_pipe)

# ======================================
# 📊 Step 3. 评估指标
# 可选指标有：
#   - faithfulness：答案是否忠实于上下文
#   - answer_relevance：答案是否回答了问题
#   - context_precision：检索是否相关
#   - context_recall：检索是否覆盖答案
# ======================================
metrics = [faithfulness, answer_relevance, context_precision, context_recall]

# ======================================
# 🚀 Step 4. 运行评估
# ======================================
results = evaluate(dataset=dataset, metrics=metrics, llm=llm)

print("\n📈 评估结果：")
for k, v in results.items():
    print(f"{k:20s}: {v:.3f}")

# ======================================
# 🧾 Step 5. 结果可视化（可选）
# ======================================
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame([results])
df.plot(kind="bar", figsize=(8,4))
plt.title("RAGAS Evaluation Results")
plt.xticks([])
plt.ylabel("Score")
plt.show()
